# Anveshak: Open-World Remote Sensing Object Detector
### Techfest 2026-27 | IITB-HRTI Centre of Excellence
---
This turnkey notebook runs GPU training, continual few-shot learning, and evaluation directly inside Google Colab.
- **Dataset (20 GB):** Automatically downloaded from the competition link directly inside Colab (no need to upload anything from your PC!).
- **Outputs:** Model checkpoints and `predictions.csv` are automatically saved to your Google Drive.

## 1. Verify GPU & Environment Setup

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

## 2. Mount Google Drive
Mount your Drive so your trained checkpoints and final predictions are saved permanently.

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_ANVESHAK_DIR = '/content/drive/MyDrive/Anveshak'
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE_ANVESHAK_DIR}/submissions", exist_ok=True)
print("[✓] Google Drive connected successfully. Checkpoints will be saved here.")

## 3. Load Project Codebase
You can load the code in **any** of these simple ways:
- **Method A (Zero Setup):** Run this cell and upload `anveshak_code.zip` (it is only 21 KB from your project folder).
- **Method B (GitHub):** Paste your repository URL below if you pushed to GitHub.

In [ ]:
import os
import zipfile

# Set GitHub repository
GITHUB_REPO_URL = "https://github.com/Likhitha-talent/IITB-Anveshak.git"
REPO_NAME = "IITB-Anveshak"

%cd /content
if GITHUB_REPO_URL.strip():
    if not os.path.exists(REPO_NAME):
        !git clone {GITHUB_REPO_URL} {REPO_NAME}
    else:
        %cd {REPO_NAME}
        !git pull
    %cd /content/{REPO_NAME}
elif os.path.exists("/content/anveshak_code.zip"):
    print("Unpacking uploaded anveshak_code.zip...")
    !mkdir -p /content/IITBBB
    with zipfile.ZipFile("/content/anveshak_code.zip", 'r') as z:
        z.extractall("/content/IITBBB")
    %cd /content/IITBBB
elif os.path.exists(f"{DRIVE_ANVESHAK_DIR}/IITBBB"):
    print("Loading code from Google Drive...")
    %cd {DRIVE_ANVESHAK_DIR}/IITBBB
else:
    # Prompt file upload for anveshak_code.zip (only ~21 KB!)
    from google.colab import files
    print("Please select and upload 'anveshak_code.zip' from your PC (Desktop/IITBBB/):")
    uploaded = files.upload()
    for fname in uploaded.keys():
        if fname.endswith(".zip"):
            !mkdir -p /content/IITBBB
            with zipfile.ZipFile(fname, 'r') as z:
                z.extractall("/content/IITBBB")
    %cd /content/IITBBB

!pip install -r requirements.txt -q
!pip install gdown -q
print("[✓] Environment and codebase ready in:", os.getcwd())

## 4. Download & Extract Round 1 Dataset (20 GB) Directly to Colab SSD
Using Google's internal cloud connection, `round1.zip` downloads in ~1-2 minutes straight from the organizer's link into Colab NVMe SSD (`/content/data`).

In [ ]:
import os
import zipfile

!mkdir -p /content/data
DATASET_FILE_ID = "1A24PTWWji509iDwf50EkJuTcwyb13RDH"
ZIP_PATH = "/content/data/round1.zip"

# 1. Download round1.zip directly at 100-200 MB/s from the competition link
if not os.path.exists(ZIP_PATH):
    print(f"[*] Downloading round1.zip (20 GB) directly from competition link...")
    !gdown {DATASET_FILE_ID} -O {ZIP_PATH}

# 2. Extract directly into fast local SSD
print("[*] Extracting round1.zip into /content/data/...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall("/content/data/")

print("[✓] Round 1 Dataset unpacked successfully:")
!ls -lh /content/data

## 5. Train Detector on Set A (From Scratch - Random Initialization)

In [ ]:
# Execute Set A training script
!python -m src.training.train_set_a \
    --config configs/base_config.yaml \
    --data_dir /content/data \
    --output_dir checkpoints/

# Back up Set A weights to Google Drive immediately
!cp checkpoints/set_a_best.pth {DRIVE_ANVESHAK_DIR}/checkpoints/

## 6. Continual Few-Shot Learning on Set B
Continues the *exact same model lineage* on Set B (10 new classes) with distillation loss and cosine classifier margin to retain Set A classes.

In [ ]:
!python -m src.training.train_set_b \
    --config configs/base_config.yaml \
    --checkpoint_in checkpoints/set_a_best.pth \
    --data_dir /content/data \
    --output_checkpoint checkpoints/round1_final.pth

# Sync final checkpoint to Google Drive
!cp checkpoints/round1_final.pth {DRIVE_ANVESHAK_DIR}/checkpoints/
print("[✓] Round 1 Checkpoint saved and backed up to Google Drive.")

## 7. Inference & Official Conflict Rule Resolution
Generates raw predictions on unlabelled Round 1 test images, applies conflict filtering (IoU >= 0.5 highest score rule), and exports `predictions.csv`.

In [ ]:
!python -m src.inference.predict \
    --checkpoint checkpoints/round1_final.pth \
    --test_images_dir /content/data/test_images \
    --output_csv submission/predictions.csv

# Copy predictions to Google Drive
!cp submission/predictions.csv {DRIVE_ANVESHAK_DIR}/submissions/

## 8. Validate Predictions with Organizer's Script

In [ ]:
# Run validate_predictions.py
!python validate_predictions.py submission/predictions.csv